# **Assignment 05: MLLM**

**Available:** Sep 16, 2025 3:00pm until Sep 30, 2025 11:59pm

**Details**
- https://huggingface.co/datasets/AI4Math/MathVistaLinks to an external site.​
- Use test set​
- Add a lora to InternVL3 and SophiaVL-R1​
- Train both loras with testmini​
- Evaluate on test
- To get results on test set​, you need to run the leaderboard, 
- instructions are here: https://mathvista.github.io/#leaderboard
- Insights on why either IVL or SVL is better in the above two runs​
- Reports, code, video and insights

## Setup

In [2]:
## Import Libraries

# Set CUDA_VISIBLE_DEVICES to make both GPUs visible
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'

# Install all packages for from the requirements.txt
%pip install -r requirements.txt

import torch
import torch.nn as nn
import torchvision
import datasets
import cv2
import matplotlib.pyplot as plt
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch import optim
from tqdm.notebook import tqdm
from torchinfo import summary
import einops
import PIL
import numpy as np
import pandas as pd
# Use a pipeline as a high-level helper
from transformers import pipeline
import time
import psutil
import gc
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import json
from collections import defaultdict
import numpy as np

# Authorize Huggingface account
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv('/mnt/Storage02/SoftwareDev/CAP_6411_Assignments/.env')

# Get Hugging Face token
hf_token = os.getenv('HUGGINGFACE_HUB_TOKEN') or os.getenv('HF_TOKEN')

if hf_token:
    print("Found Hugging Face token in environment variables")
    
    
    from huggingface_hub import login, whoami
    
    try:
        # Login to Hugging Face Hub
        login(token=hf_token)
        
        # Verify login by getting user info
        user_info = whoami()
        print(f"Successfully authenticated with Hugging Face!")
        print(f"Logged in as: {user_info['name']}")
        
        # Set the token as environment variable for other libraries
        os.environ['HUGGINGFACE_HUB_TOKEN'] = hf_token
        os.environ['HF_TOKEN'] = hf_token
        
    except Exception as e:
        print(f"Authentication failed: {e}")
        print("Will proceed without pre-trained models if needed")
        hf_token = None
else:
    print("No Hugging Face token found in .env file")
    print("Please add HUGGINGFACE_HUB_TOKEN=your_token_here to your .env file")
    hf_token = None


# If no logs folder exists, create one
if not os.path.exists("logs"):
    os.makedirs("logs")

# If no checkpoints folder exists, create one
if not os.path.exists("checkpoints"):
    os.makedirs("checkpoints")

# If no data folder exists, create one
if not os.path.exists("data"):
    os.makedirs("data")


Note: you may need to restart the kernel to use updated packages.


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Found Hugging Face token in environment variables
Successfully authenticated with Hugging Face!
Logged in as: malneyugnfl


In [4]:
# GPU Setup 
# Comprehensive GPU diagnostics
print("\n=== GPU Diagnostics ===")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available: {torch.backends.mps.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"Number of GPUs detected: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    print("\n=== All Available GPUs ===")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i}:")
        print(f"  Name: {props.name}")
        print(f"  Total Memory: {props.total_memory / 1024**3:.2f} GB")
        print(f"  Multi-processor count: {props.multi_processor_count}")
        print(f"  Compute Capability: {props.major}.{props.minor}")
        print()

# Device selection with preference for cuda:1 (A6000) -> cuda:0 (4090) -> mps (Apple Silicon) -> cpu
if torch.cuda.is_available() and torch.cuda.device_count() > 1:
    device = torch.device('cuda:1')  # This should now be your A6000!
    print(f"Using GPU 1: {torch.cuda.get_device_name(1)}")
elif torch.cuda.is_available():
    device = torch.device('cuda:0')
    print(f"Using GPU 0: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device('mps')
    print("Using Apple Silicon MPS")
else:
    device = torch.device('cpu')
    print("Using CPU")

print(f"Selected device: {device}")

# If no logs folder exists, create one
if not os.path.exists("logs"):
    os.makedirs("logs")

# If no checkpoints folder exists, create one
if not os.path.exists("checkpoints"):
    os.makedirs("checkpoints")

# If no data folder exists, create one
if not os.path.exists("data"):
    os.makedirs("data")

def get_memory_usage():
    """Get current memory usage in MB"""
    process = psutil.Process()
    return process.memory_info().rss / 1024 / 1024

def get_gpu_memory_usage():
    """Get current GPU memory usage in MB"""
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1024 / 1024
    elif device.type == 'mps':
        # MPS doesn't have direct memory monitoring like CUDA
        # Return 0 as a placeholder
        return 0
    return 0


=== GPU Diagnostics ===
PyTorch version: 2.8.0+cu128
CUDA available: True
MPS available: False
CUDA version: 12.8
Number of GPUs detected: 1

=== All Available GPUs ===
GPU 0:
  Name: NVIDIA GeForce RTX 4090 Laptop GPU
  Total Memory: 15.70 GB
  Multi-processor count: 76
  Compute Capability: 8.9

Using GPU 0: NVIDIA GeForce RTX 4090 Laptop GPU
Selected device: cuda:0


In [5]:
# Function to clear memory for both CUDA and MPS
def clear_memory():
    """Clear CPU and GPU memory"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    elif torch.backends.mps.is_available():
        torch.mps.empty_cache()
    print(f"Cleared memory. Current CPU memory usage: {get_memory_usage():.2f} MB, GPU memory usage: {get_gpu_memory_usage():.2f} MB")

## Data Preparation and Processing

In [6]:
# Import the Dataset
# Source: https://huggingface.co/datasets/AI4Math/MathVista

from datasets import load_dataset

dataset = load_dataset("AI4Math/MathVista")

In [7]:
# Examine the dataset structure
print("Dataset structure:")
print(dataset)
print("\nDataset keys:")
print(dataset.keys())

# Check the structure of each split
for split_name in dataset.keys():
    print(f"\n{split_name} split:")
    print(f"  Number of examples: {len(dataset[split_name])}")
    if len(dataset[split_name]) > 0:
        print(f"  Features: {dataset[split_name].features}")
        print(f"  First example keys: {list(dataset[split_name][0].keys())}")
        
        # Show a sample of the first example
        sample = dataset[split_name][0]
        print(f"  Sample data:")
        for key, value in sample.items():
            if key == 'image' and value is not None:
                print(f"    {key}: PIL Image ({value.size if hasattr(value, 'size') else 'unknown size'})")
            elif isinstance(value, str) and len(value) > 100:
                print(f"    {key}: {value[:100]}...")
            else:
                print(f"    {key}: {value}")
        break

Dataset structure:
DatasetDict({
    testmini: Dataset({
        features: ['pid', 'question', 'image', 'decoded_image', 'choices', 'unit', 'precision', 'answer', 'question_type', 'answer_type', 'metadata', 'query'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['pid', 'question', 'image', 'decoded_image', 'choices', 'unit', 'precision', 'answer', 'question_type', 'answer_type', 'metadata', 'query'],
        num_rows: 5141
    })
})

Dataset keys:
dict_keys(['testmini', 'test'])

testmini split:
  Number of examples: 1000
  Features: {'pid': Value('string'), 'question': Value('string'), 'image': Value('string'), 'decoded_image': Image(mode=None, decode=True), 'choices': List(Value('string')), 'unit': Value('string'), 'precision': Value('float64'), 'answer': Value('string'), 'question_type': Value('string'), 'answer_type': Value('string'), 'metadata': {'category': Value('string'), 'context': Value('string'), 'grade': Value('string'), 'img_height': Value('int64'), 

In [9]:
# Data Processing Functions
import json
from typing import Dict, List, Any, Tuple
from PIL import Image
import io
import base64

class MathVistaProcessor:
    def __init__(self):
        self.processed_data = {
            'testmini': [],
            'test': []
        }
    
    def format_question_for_models(self, example: Dict) -> Tuple[str, Image.Image]:
        """Format question and image for model input"""
        # Get the image
        image = example['decoded_image']
        
        # Format the question based on question type
        question = example['question']
        query = example.get('query', '')
        
        # Add context based on answer type
        if example['answer_type'] == 'float':
            if example.get('precision'):
                formatted_question = f"{question}\n\nPlease provide your answer as a floating-point number with {int(example['precision'])} decimal place(s)."
            else:
                formatted_question = f"{question}\n\nPlease provide your answer as a floating-point number."
        elif example['answer_type'] == 'integer':
            formatted_question = f"{question}\n\nPlease provide your answer as an integer."
        elif example['question_type'] == 'multi_choice':
            if example.get('choices'):
                choices_text = '\n'.join([f"{i+1}. {choice}" for i, choice in enumerate(example['choices'])])
                formatted_question = f"{question}\n\nChoices:\n{choices_text}\n\nPlease select the correct answer."
            else:
                formatted_question = question
        else:
            formatted_question = question
        
        # Add any additional query information
        if query and query != question:
            formatted_question = f"{formatted_question}\n\nAdditional context: {query}"
        
        return formatted_question, image
    
    def process_split(self, split_name: str, max_samples: int = None) -> List[Dict]:
        """Process a specific split of the dataset"""
        split_data = dataset[split_name]
        processed_examples = []
        
        print(f"Processing {split_name} split...")
        
        # Limit samples if specified
        if max_samples:
            split_data = split_data.select(range(min(max_samples, len(split_data))))
        
        for idx, example in enumerate(tqdm(split_data, desc=f"Processing {split_name}")):
            try:
                # Format question and get image
                formatted_question, image = self.format_question_for_models(example)
                
                # Create processed example
                processed_example = {
                    'pid': example['pid'],
                    'formatted_question': formatted_question,
                    'original_question': example['question'],
                    'image': image,
                    'ground_truth_answer': example['answer'],
                    'question_type': example['question_type'],
                    'answer_type': example['answer_type'],
                    'choices': example.get('choices'),
                    'unit': example.get('unit'),
                    'precision': example.get('precision'),
                    'metadata': example['metadata'],
                    'query': example.get('query', ''),
                }
                
                processed_examples.append(processed_example)
                
                # Print first example for verification
                if idx == 0:
                    print(f"\nFirst example from {split_name}:")
                    print(f"PID: {processed_example['pid']}")
                    print(f"Question type: {processed_example['question_type']}")
                    print(f"Answer type: {processed_example['answer_type']}")
                    print(f"Formatted question: {processed_example['formatted_question'][:200]}...")
                    print(f"Ground truth: {processed_example['ground_truth_answer']}")
                    print(f"Image size: {processed_example['image'].size}")
                    print("-" * 50)
                
            except Exception as e:
                print(f"Error processing example {idx} in {split_name}: {e}")
                continue
        
        self.processed_data[split_name] = processed_examples
        print(f"Successfully processed {len(processed_examples)} examples from {split_name}")
        
        return processed_examples
    
    def get_statistics(self):
        """Get statistics about the processed data"""
        stats = {}
        
        for split_name, examples in self.processed_data.items():
            if not examples:
                continue
                
            stats[split_name] = {
                'total_examples': len(examples),
                'question_types': {},
                'answer_types': {},
                'categories': {},
            }
            
            for example in examples:
                # Question types
                q_type = example['question_type']
                stats[split_name]['question_types'][q_type] = stats[split_name]['question_types'].get(q_type, 0) + 1
                
                # Answer types
                a_type = example['answer_type']
                stats[split_name]['answer_types'][a_type] = stats[split_name]['answer_types'].get(a_type, 0) + 1
                
                # Categories
                category = example['metadata']['category']
                stats[split_name]['categories'][category] = stats[split_name]['categories'].get(category, 0) + 1
        
        return stats

# Initialize processor
processor = MathVistaProcessor()

print("MathVista processor initialized successfully!")

MathVista processor initialized successfully!


In [10]:
# Process the testmini split (for training LoRA)
print("Processing testmini split for training...")
testmini_processed = processor.process_split('testmini')

print(f"\nTestmini processing completed!")
print(f"Total examples processed: {len(testmini_processed)}")

# Show some statistics
testmini_stats = processor.get_statistics()
print("\nTestmini Statistics:")
print(f"Question types: {testmini_stats['testmini']['question_types']}")
print(f"Answer types: {testmini_stats['testmini']['answer_types']}")
print(f"Categories: {list(testmini_stats['testmini']['categories'].keys())}")

Processing testmini split for training...
Processing testmini split...


Processing testmini:   0%|          | 0/1000 [00:00<?, ?it/s]


First example from testmini:
PID: 1
Question type: free_form
Answer type: float
Formatted question: When a spring does work on an object, we cannot find the work by simply multiplying the spring force by the object's displacement. The reason is that there is no one value for the force-it changes. Ho...
Ground truth: 1.2
Image size: (1514, 720)
--------------------------------------------------
Successfully processed 1000 examples from testmini

Testmini processing completed!
Total examples processed: 1000

Testmini Statistics:
Question types: {'free_form': 460, 'multi_choice': 540}
Answer types: {'float': 40, 'integer': 418, 'text': 540, 'list': 2}
Categories: ['math-targeted-vqa', 'general-vqa']


In [11]:
# Process the test split (for evaluation)
print("Processing test split for evaluation...")
test_processed = processor.process_split('test')

print(f"\nTest processing completed!")
print(f"Total examples processed: {len(test_processed)}")

# Show complete statistics
all_stats = processor.get_statistics()
print("\n" + "="*60)
print("COMPLETE DATASET STATISTICS")
print("="*60)

for split_name, stats in all_stats.items():
    print(f"\n{split_name.upper()} SPLIT:")
    print(f"  Total examples: {stats['total_examples']}")
    print(f"  Question types: {stats['question_types']}")
    print(f"  Answer types: {stats['answer_types']}")
    print(f"  Categories: {stats['categories']}")
    print("-" * 40)

Processing test split for evaluation...
Processing test split...


Processing test:   0%|          | 0/5141 [00:00<?, ?it/s]


First example from test:
PID: 1001
Question type: free_form
Answer type: integer
Formatted question: In how many years, is the percentage of labor tax greater than 3 %?

Please provide your answer as an integer.

Additional context: Hint: Please answer the question requiring an integer answer and pro...
Ground truth: 
Image size: (981, 650)
--------------------------------------------------
Successfully processed 5141 examples from test

Test processing completed!
Total examples processed: 5141

COMPLETE DATASET STATISTICS

TESTMINI SPLIT:
  Total examples: 1000
  Question types: {'free_form': 460, 'multi_choice': 540}
  Answer types: {'float': 40, 'integer': 418, 'text': 540, 'list': 2}
  Categories: {'math-targeted-vqa': 540, 'general-vqa': 460}
----------------------------------------

TEST SPLIT:
  Total examples: 5141
  Question types: {'free_form': 2289, 'multi_choice': 2852}
  Answer types: {'integer': 2043, 'text': 2852, 'float': 232, 'list': 14}
  Categories: {'general-vqa': 

In [12]:
# Utility functions to access processed data
def get_training_data():
    """Get processed testmini data for training LoRA"""
    return processor.processed_data['testmini']

def get_evaluation_data():
    """Get processed test data for evaluation"""
    return processor.processed_data['test']

def get_sample_batch(split='testmini', batch_size=4, start_idx=0):
    """Get a sample batch for testing model inference"""
    if split not in processor.processed_data:
        print(f"Split '{split}' not found. Available splits: {list(processor.processed_data.keys())}")
        return []
    
    data = processor.processed_data[split]
    end_idx = min(start_idx + batch_size, len(data))
    return data[start_idx:end_idx]

def save_processed_data(filename_prefix="mathvista_processed"):
    """Save processed data to files for later use"""
    import pickle
    
    # Save testmini data
    with open(f"data/{filename_prefix}_testmini.pkl", 'wb') as f:
        pickle.dump(processor.processed_data['testmini'], f)
    print(f"Testmini data saved to data/{filename_prefix}_testmini.pkl")
    
    # Save test data
    with open(f"data/{filename_prefix}_test.pkl", 'wb') as f:
        pickle.dump(processor.processed_data['test'], f)
    print(f"Test data saved to data/{filename_prefix}_test.pkl")
    
    # Save statistics
    stats = processor.get_statistics()
    with open(f"data/{filename_prefix}_stats.json", 'w') as f:
        json.dump(stats, f, indent=2)
    print(f"Statistics saved to data/{filename_prefix}_stats.json")

def load_processed_data(filename_prefix="mathvista_processed"):
    """Load previously processed data"""
    import pickle
    
    try:
        # Load testmini data
        with open(f"data/{filename_prefix}_testmini.pkl", 'rb') as f:
            processor.processed_data['testmini'] = pickle.load(f)
        print(f"Testmini data loaded from data/{filename_prefix}_testmini.pkl")
        
        # Load test data
        with open(f"data/{filename_prefix}_test.pkl", 'rb') as f:
            processor.processed_data['test'] = pickle.load(f)
        print(f"Test data loaded from data/{filename_prefix}_test.pkl")
        
        return True
    except FileNotFoundError as e:
        print(f"Could not load processed data: {e}")
        return False

# Save the processed data
save_processed_data()

print("\n" + "="*60)
print("DATASET PROCESSING COMPLETED SUCCESSFULLY!")
print("="*60)
print(f"✓ Testmini split: {len(processor.processed_data['testmini'])} examples (for LoRA training)")
print(f"✓ Test split: {len(processor.processed_data['test'])} examples (for evaluation)")
print("✓ Data formatted for both InternVL3 and SophiaVL-R1 models")
print("✓ Processed data saved to pickle files")
print("\nYou can now proceed with:")
print("1. Training LoRA adapters on both models using testmini data")
print("2. Evaluating both models on test data")
print("3. Comparing performance between InternVL3 and SophiaVL-R1")

Testmini data saved to data/mathvista_processed_testmini.pkl
Test data saved to data/mathvista_processed_test.pkl
Statistics saved to data/mathvista_processed_stats.json

DATASET PROCESSING COMPLETED SUCCESSFULLY!
✓ Testmini split: 1000 examples (for LoRA training)
✓ Test split: 5141 examples (for evaluation)
✓ Data formatted for both InternVL3 and SophiaVL-R1 models
✓ Processed data saved to pickle files

You can now proceed with:
1. Training LoRA adapters on both models using testmini data
2. Evaluating both models on test data
3. Comparing performance between InternVL3 and SophiaVL-R1


## Import InternVL3 Model

In [16]:
# Source: https://huggingface.co/OpenGVLab/InternVL3-78B

# Load model directly
from transformers import AutoModel
from peft import LoraConfig, get_peft_model, TaskType

model = AutoModel.from_pretrained("OpenGVLab/InternVL3-78B", trust_remote_code=True, dtype="auto")

# Define LoRA configuration for InternVL3-78B
lora_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,  # Or use TaskType.SEQ_CLS if it's for classification
    inference_mode=False,  # Set to True for inference only
    r=16,  # Rank of the adaptation
    lora_alpha=32,  # LoRA scaling parameter
    lora_dropout=0.1,  # Dropout for LoRA layers
    target_modules=[  # Target specific modules in InternVL3
        # Vision encoder targets (QFormer/ViT)
        "query",
        "key", 
        "value",
        "dense",
        # Language model targets (LLaMA)
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj", 
        "down_proj",
        # Cross-attention layers
        "cross_attention",
        "encoder_attn",
    ],
    bias="none",
    modules_to_save=["classifier", "score"]  # Save these modules fully if they exist
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)


Loading checkpoint shards:   0%|          | 0/33 [00:00<?, ?it/s]

## Fine Tune Intern VL3 Model

In [14]:
# Fine Tune InternVL3 Model with LoRA on TestMini Subset
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, TrainingArguments, Trainer
from transformers.integrations import TensorBoardCallback
from PIL import Image
import json
import os
from datetime import datetime
import wandb
from sklearn.metrics import accuracy_score, f1_score
import re
import logging
import sys
from pathlib import Path

# Custom Dataset class for MathVista
class MathVistaDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=512):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        example = self.data[idx]
        
        # Format the input with question and image
        question = example['formatted_question']
        image = example['image']
        ground_truth = str(example['ground_truth_answer'])
        
        # Create conversation format expected by InternVL3
        conversation = [
            {
                "role": "user",
                "content": f"<image>\n{question}"
            },
            {
                "role": "assistant", 
                "content": ground_truth
            }
        ]
        
        return {
            'conversation': conversation,
            'image': image,
            'ground_truth': ground_truth,
            'pid': example['pid'],
            'question_type': example['question_type'],
            'answer_type': example['answer_type']
        }

# Custom collate function for batching
def collate_fn(batch):
    conversations = [item['conversation'] for item in batch]
    images = [item['image'] for item in batch]
    ground_truths = [item['ground_truth'] for item in batch]
    pids = [item['pid'] for item in batch]
    question_types = [item['question_type'] for item in batch]
    answer_types = [item['answer_type'] for item in batch]
    
    return {
        'conversations': conversations,
        'images': images,
        'ground_truths': ground_truths,
        'pids': pids,
        'question_types': question_types,
        'answer_types': answer_types
    }

# Enhanced Logging Setup
def setup_logging(log_dir="./logs", experiment_name="internvl3_training"):
    """Setup comprehensive logging for training and evaluation"""
    
    # Create timestamp for unique log files
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Create log directory if it doesn't exist
    Path(log_dir).mkdir(parents=True, exist_ok=True)
    
    # Create experiment-specific log directory
    exp_log_dir = Path(log_dir) / f"{experiment_name}_{timestamp}"
    exp_log_dir.mkdir(parents=True, exist_ok=True)
    
    # Setup main logger
    logger = logging.getLogger('internvl3_training')
    logger.setLevel(logging.INFO)
    
    # Clear any existing handlers
    logger.handlers.clear()
    
    # Create formatters
    detailed_formatter = logging.Formatter(
        '%(asctime)s - %(name)s - %(levelname)s - %(funcName)s:%(lineno)d - %(message)s'
    )
    simple_formatter = logging.Formatter(
        '%(asctime)s - %(levelname)s - %(message)s'
    )
    
    # File handler for detailed logs
    detailed_log_file = exp_log_dir / "training_detailed.log"
    file_handler = logging.FileHandler(detailed_log_file)
    file_handler.setLevel(logging.DEBUG)
    file_handler.setFormatter(detailed_formatter)
    logger.addHandler(file_handler)
    
    # File handler for important events only
    important_log_file = exp_log_dir / "training_summary.log"
    summary_handler = logging.FileHandler(important_log_file)
    summary_handler.setLevel(logging.INFO)
    summary_handler.setFormatter(simple_formatter)
    logger.addHandler(summary_handler)
    
    # Console handler
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(logging.INFO)
    console_handler.setFormatter(simple_formatter)
    logger.addHandler(console_handler)
    
    # Create separate loggers for different components
    setup_component_loggers(exp_log_dir)
    
    logger.info(f"Logging setup completed. Logs will be saved to: {exp_log_dir}")
    
    return logger, exp_log_dir

def setup_component_loggers(log_dir):
    """Setup loggers for different components"""
    
    components = ['model', 'data', 'training', 'evaluation', 'metrics']
    
    for component in components:
        comp_logger = logging.getLogger(f'internvl3_{component}')
        comp_logger.setLevel(logging.INFO)
        
        # Clear existing handlers
        comp_logger.handlers.clear()
        
        # File handler for component
        comp_log_file = log_dir / f"{component}.log"
        comp_handler = logging.FileHandler(comp_log_file)
        comp_handler.setLevel(logging.INFO)
        
        formatter = logging.Formatter(
            '%(asctime)s - %(levelname)s - %(message)s'
        )
        comp_handler.setFormatter(formatter)
        comp_logger.addHandler(comp_handler)
        
        # Prevent propagation to avoid duplicate logs
        comp_logger.propagate = False

def log_system_info():
    """Log system information"""
    logger = logging.getLogger('internvl3_training')
    
    logger.info("="*60)
    logger.info("SYSTEM INFORMATION")
    logger.info("="*60)
    
    # GPU Information
    logger.info(f"PyTorch version: {torch.__version__}")
    logger.info(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        logger.info(f"CUDA version: {torch.version.cuda}")
        logger.info(f"Number of GPUs: {torch.cuda.device_count()}")
        for i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(i)
            logger.info(f"GPU {i}: {props.name} ({props.total_memory / 1024**3:.2f} GB)")
    
    # Memory Information
    logger.info(f"CPU Memory: {get_memory_usage():.2f} MB")
    logger.info(f"GPU Memory: {get_gpu_memory_usage():.2f} MB")
    
    logger.info("="*60)

def log_training_config(config):
    """Log training configuration"""
    logger = logging.getLogger('internvl3_training')
    
    logger.info("="*60)
    logger.info("TRAINING CONFIGURATION")
    logger.info("="*60)
    
    training_args = config['training_args']
    
    logger.info(f"Output directory: {training_args.output_dir}")
    logger.info(f"Number of epochs: {training_args.num_train_epochs}")
    logger.info(f"Train batch size: {training_args.per_device_train_batch_size}")
    logger.info(f"Eval batch size: {training_args.per_device_eval_batch_size}")
    logger.info(f"Gradient accumulation steps: {training_args.gradient_accumulation_steps}")
    logger.info(f"Learning rate: {training_args.learning_rate}")
    logger.info(f"Weight decay: {training_args.weight_decay}")
    logger.info(f"Warmup ratio: {training_args.warmup_ratio}")
    logger.info(f"FP16: {training_args.fp16}")
    logger.info(f"Gradient checkpointing: {training_args.gradient_checkpointing}")
    
    # Dataset information
    logger.info(f"Training examples: {len(config['train_dataset'])}")
    logger.info(f"Validation examples: {len(config['val_dataset'])}")
    
    logger.info("="*60)

# Custom Trainer class with enhanced logging
class InternVL3Trainer(Trainer):
    def __init__(self, model, tokenizer, log_dir=None, **kwargs):
        super().__init__(**kwargs)
        self.model = model
        self.tokenizer = tokenizer
        self.log_dir = log_dir
        self.training_logger = logging.getLogger('internvl3_training')
        self.model_logger = logging.getLogger('internvl3_model')
        
        # Initialize metrics tracking
        self.training_metrics = {
            'epoch_losses': [],
            'step_losses': [],
            'learning_rates': [],
            'validation_losses': [],
            'timestamps': []
        }
        
    def log_metrics_to_file(self, logs, step=None, epoch=None):
        """Save metrics to JSON file"""
        if self.log_dir:
            metrics_file = Path(self.log_dir) / "training_metrics.json"
            
            # Update metrics
            if 'loss' in logs:
                self.training_metrics['step_losses'].append({
                    'step': step,
                    'epoch': epoch,
                    'loss': logs['loss'],
                    'timestamp': datetime.now().isoformat()
                })
            
            if 'learning_rate' in logs:
                self.training_metrics['learning_rates'].append({
                    'step': step,
                    'epoch': epoch,
                    'lr': logs['learning_rate'],
                    'timestamp': datetime.now().isoformat()
                })
            
            if 'eval_loss' in logs:
                self.training_metrics['validation_losses'].append({
                    'step': step,
                    'epoch': epoch,
                    'eval_loss': logs['eval_loss'],
                    'timestamp': datetime.now().isoformat()
                })
            
            # Save to file
            with open(metrics_file, 'w') as f:
                json.dump(self.training_metrics, f, indent=2)
        
    def log(self, logs):
        """Override log method to add custom logging"""
        super().log(logs)
        
        # Log to our custom loggers
        step = logs.get('step', 'N/A')
        epoch = logs.get('epoch', 'N/A')
        
        if 'loss' in logs:
            self.training_logger.info(f"Step {step}, Epoch {epoch:.2f}: Loss = {logs['loss']:.6f}")
            
        if 'learning_rate' in logs:
            self.training_logger.info(f"Step {step}: Learning Rate = {logs['learning_rate']:.2e}")
            
        if 'eval_loss' in logs:
            self.training_logger.info(f"Step {step}: Validation Loss = {logs['eval_loss']:.6f}")
            
        if 'grad_norm' in logs:
            self.training_logger.info(f"Step {step}: Gradient Norm = {logs['grad_norm']:.6f}")
        
        # Save metrics to file
        self.log_metrics_to_file(logs, step, epoch)
        
        # Log memory usage periodically
        if step != 'N/A' and int(step) % 10 == 0:
            cpu_mem = get_memory_usage()
            gpu_mem = get_gpu_memory_usage()
            self.model_logger.info(f"Step {step}: CPU Memory = {cpu_mem:.2f} MB, GPU Memory = {gpu_mem:.2f} MB")
    
    def on_epoch_begin(self, args, state, control, **kwargs):
        """Log epoch start"""
        self.training_logger.info(f"Starting Epoch {state.epoch}")
        return super().on_epoch_begin(args, state, control, **kwargs)
    
    def on_epoch_end(self, args, state, control, **kwargs):
        """Log epoch end"""
        self.training_logger.info(f"Completed Epoch {state.epoch}")
        
        # Clear memory at end of epoch
        clear_memory()
        
        return super().on_epoch_end(args, state, control, **kwargs)
        
    def compute_loss(self, model, inputs, return_outputs=False):
        """Custom loss computation for vision-language model"""
        try:
            # Process the batch
            conversations = inputs['conversations']
            images = inputs['images']
            
            # Format inputs for the model (this will depend on InternVL3's specific API)
            # For now, we'll implement a basic approach
            total_loss = 0
            batch_size = len(conversations)
            
            for i, (conversation, image) in enumerate(zip(conversations, images)):
                try:
                    # Generate response using the model
                    # Note: This is a simplified approach - you may need to adjust based on InternVL3's API
                    with torch.cuda.amp.autocast():
                        # Model forward pass - adjust based on actual InternVL3 API
                        outputs = model.generate(
                            image=image,
                            question=conversation[0]['content'],
                            max_new_tokens=100,
                            do_sample=False
                        )
                        
                        # Compute loss between generated and ground truth
                        # This is a placeholder - implement actual loss computation
                        loss = torch.tensor(0.0, requires_grad=True, device=model.device)
                        total_loss += loss
                        
                except Exception as e:
                    self.training_logger.error(f"Error processing sample {i}: {e}")
                    continue
            
            avg_loss = total_loss / batch_size if batch_size > 0 else torch.tensor(0.0)
            
            if return_outputs:
                return avg_loss, None
            return avg_loss
            
        except Exception as e:
            self.training_logger.error(f"Error in compute_loss: {e}")
            return torch.tensor(0.0, requires_grad=True, device=model.device)

# Setup training configuration
def setup_internvl3_training():
    """Setup training configuration for InternVL3 with LoRA"""
    
    # Setup logging first
    logger, log_dir = setup_logging("./logs", "internvl3_training")
    
    logger.info("Setting up InternVL3 training configuration...")
    
    # Log system information
    log_system_info()
    
    # Get training data
    training_data = get_training_data()
    logger.info(f"Training data loaded: {len(training_data)} examples")
    
    # Log data statistics
    data_logger = logging.getLogger('internvl3_data')
    data_logger.info(f"Total training examples: {len(training_data)}")
    
    # Analyze data distribution
    question_types = {}
    answer_types = {}
    for example in training_data:
        q_type = example['question_type']
        a_type = example['answer_type']
        question_types[q_type] = question_types.get(q_type, 0) + 1
        answer_types[a_type] = answer_types.get(a_type, 0) + 1
    
    data_logger.info(f"Question type distribution: {question_types}")
    data_logger.info(f"Answer type distribution: {answer_types}")
    
    # Create dataset
    tokenizer = None  # InternVL3 may not use a standard tokenizer
    train_dataset = MathVistaDataset(training_data, tokenizer)
    
    # Split training data for validation (80-20 split)
    train_size = int(0.8 * len(training_data))
    val_size = len(training_data) - train_size
    train_subset = training_data[:train_size]
    val_subset = training_data[train_size:]
    
    train_dataset = MathVistaDataset(train_subset, tokenizer)
    val_dataset = MathVistaDataset(val_subset, tokenizer)
    
    print(f"Training set: {len(train_dataset)} examples")
    print(f"Validation set: {len(val_dataset)} examples")
    
    # Create data loaders
    train_dataloader = DataLoader(
        train_dataset,
        batch_size=2,  # Small batch size due to large model
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=2
    )
    
    val_dataloader = DataLoader(
        val_dataset,
        batch_size=2,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=2
    )
    
    # Training arguments
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_dir = f"./checkpoints/internvl3_lora_{timestamp}"
    
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=3,
        per_device_train_batch_size=1,  # Very small due to model size
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=8,  # Effective batch size = 8
        warmup_ratio=0.1,
        learning_rate=5e-5,
        weight_decay=0.01,
        logging_dir=f"./logs/internvl3_lora_{timestamp}",
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        dataloader_pin_memory=False,
        fp16=True,  # Use mixed precision
        gradient_checkpointing=True,  # Save memory
        remove_unused_columns=False,
        report_to=["tensorboard"],
        run_name=f"internvl3_lora_{timestamp}",
    )
    
    # Log training configuration
    config = {
        'train_dataset': train_dataset,
        'val_dataset': val_dataset,
        'train_dataloader': train_dataloader,
        'val_dataloader': val_dataloader,
        'training_args': training_args,
        'output_dir': output_dir,
        'log_dir': log_dir
    }
    
    log_training_config(config)
    
    return config

# Evaluation functions
def evaluate_model_on_mathvista(model, tokenizer, data, model_name="InternVL3", log_dir=None):
    """Evaluate model on MathVista data"""
    
    # Setup evaluation logging
    eval_logger = logging.getLogger('internvl3_evaluation')
    if not eval_logger.handlers:
        eval_logger.setLevel(logging.INFO)
        handler = logging.StreamHandler()
        formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
        handler.setFormatter(formatter)
        eval_logger.addHandler(handler)
        
        # Add file handler if log_dir is provided
        if log_dir:
            file_handler = logging.FileHandler(Path(log_dir) / 'evaluation.log')
            file_handler.setFormatter(formatter)
            eval_logger.addHandler(file_handler)
    
    eval_logger.info(f"Starting evaluation of {model_name} on {len(data)} examples...")
    
    results = []
    correct_predictions = 0
    total_predictions = 0
    
    # Track evaluation metrics
    evaluation_start_time = datetime.now()
    
    model.eval()
    
    with torch.no_grad():
        for i, example in enumerate(tqdm(data, desc=f"Evaluating {model_name}")):
            try:
                question = example['formatted_question']
                image = example['image']
                ground_truth = str(example['ground_truth_answer'])
                
                # Generate prediction (adjust based on InternVL3 API)
                try:
                    prediction = model.generate(
                        image=image,
                        question=question,
                        max_new_tokens=50,
                        do_sample=False
                    )
                    
                    # Extract answer from prediction
                    predicted_answer = extract_answer(prediction, example['answer_type'])
                    
                    # Check if prediction matches ground truth
                    is_correct = compare_answers(predicted_answer, ground_truth, example['answer_type'])
                    
                    if is_correct:
                        correct_predictions += 1
                    total_predictions += 1
                    
                    # Store result
                    result = {
                        'pid': example['pid'],
                        'question': question,
                        'ground_truth': ground_truth,
                        'prediction': predicted_answer,
                        'correct': is_correct,
                        'question_type': example['question_type'],
                        'answer_type': example['answer_type']
                    }
                    results.append(result)
                    
                    if i % 50 == 0:
                        current_accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0
                        print(f"Progress: {i+1}/{len(data)}, Current accuracy: {current_accuracy:.3f}")
                        
                except Exception as e:
                    print(f"Error generating prediction for example {i}: {e}")
                    continue
                    
            except Exception as e:
                print(f"Error processing example {i}: {e}")
                continue
    
    # Calculate final metrics
    accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0
    
    print(f"\n{model_name} Evaluation Results:")
    print(f"Total examples processed: {total_predictions}")
    print(f"Correct predictions: {correct_predictions}")
    print(f"Accuracy: {accuracy:.4f}")
    
    # Calculate accuracy by question type
    question_type_accuracy = {}
    for q_type in set([r['question_type'] for r in results]):
        type_results = [r for r in results if r['question_type'] == q_type]
        type_correct = sum([r['correct'] for r in type_results])
        type_accuracy = type_correct / len(type_results) if type_results else 0
        question_type_accuracy[q_type] = type_accuracy
        print(f"  {q_type}: {type_accuracy:.4f} ({type_correct}/{len(type_results)})")
    
    return {
        'results': results,
        'accuracy': accuracy,
        'question_type_accuracy': question_type_accuracy,
        'total_processed': total_predictions,
        'correct_predictions': correct_predictions
    }

def extract_answer(prediction_text, answer_type):
    """Extract answer from model prediction based on answer type"""
    
    if answer_type == 'integer':
        # Look for integers in the text
        numbers = re.findall(r'-?\d+', prediction_text)
        if numbers:
            return int(numbers[0])
        return None
    
    elif answer_type == 'float':
        # Look for floating point numbers
        numbers = re.findall(r'-?\d+\.?\d*', prediction_text)
        if numbers:
            try:
                return float(numbers[0])
            except ValueError:
                return None
        return None
    
    else:
        # For text answers, return the full prediction cleaned up
        return prediction_text.strip()

def compare_answers(predicted, ground_truth, answer_type):
    """Compare predicted answer with ground truth"""
    
    if predicted is None:
        return False
    
    if answer_type in ['integer', 'float']:
        try:
            pred_num = float(predicted)
            gt_num = float(ground_truth)
            # Allow small floating point errors
            return abs(pred_num - gt_num) < 1e-6
        except (ValueError, TypeError):
            return False
    
    else:
        # Text comparison (case insensitive)
        return str(predicted).lower().strip() == str(ground_truth).lower().strip()

# Training function
def train_internvl3_with_lora():
    """Train InternVL3 model with LoRA on testmini data"""
    
    # Setup training configuration (includes logging setup)
    training_config = setup_internvl3_training()
    
    # Get logger
    logger = logging.getLogger('internvl3_training')
    model_logger = logging.getLogger('internvl3_model')
    
    logger.info("="*60)
    logger.info("STARTING INTERNVL3 LORA TRAINING")
    logger.info("="*60)
    
    # Move model to device
    global model
    model = model.to(device)
    
    # Log model information
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    model_logger.info(f"Model moved to device: {device}")
    model_logger.info(f"Total model parameters: {total_params:,}")
    model_logger.info(f"Trainable parameters: {trainable_params:,}")
    model_logger.info(f"Trainable parameter ratio: {trainable_params/total_params:.4f}")
    
    # Log LoRA configuration
    if hasattr(model, 'peft_config'):
        peft_config = model.peft_config
        for adapter_name, config in peft_config.items():
            model_logger.info(f"LoRA Adapter '{adapter_name}' Configuration:")
            model_logger.info(f"  Rank (r): {config.r}")
            model_logger.info(f"  Alpha: {config.lora_alpha}")
            model_logger.info(f"  Dropout: {config.lora_dropout}")
            model_logger.info(f"  Target modules: {config.target_modules}")
    
    try:
        # Create trainer with enhanced logging
        trainer = InternVL3Trainer(
            model=model,
            tokenizer=None,
            log_dir=training_config['log_dir'],
            args=training_config['training_args'],
            train_dataset=training_config['train_dataset'],
            eval_dataset=training_config['val_dataset'],
            data_collator=collate_fn,
        )
        
        # Log training start
        training_start_time = datetime.now()
        logger.info(f"Starting training at: {training_start_time}")
        logger.info(f"Expected training time: ~{training_config['training_args'].num_train_epochs * len(training_config['train_dataset']) / (training_config['training_args'].per_device_train_batch_size * training_config['training_args'].gradient_accumulation_steps):.0f} steps")
        
        # Start training
        trainer.train()
        
        # Log training completion
        training_end_time = datetime.now()
        training_duration = training_end_time - training_start_time
        logger.info(f"Training completed at: {training_end_time}")
        logger.info(f"Total training time: {training_duration}")
        
        # Save the final model
        final_model_path = os.path.join(training_config['output_dir'], 'final_model')
        trainer.save_model(final_model_path)
        logger.info(f"Final model saved to: {final_model_path}")
        
        # Save LoRA adapter
        lora_adapter_path = os.path.join(training_config['output_dir'], 'lora_adapter')
        model.save_pretrained(lora_adapter_path)
        logger.info(f"LoRA adapter saved to: {lora_adapter_path}")
        
        # Save training summary
        training_summary = {
            'start_time': training_start_time.isoformat(),
            'end_time': training_end_time.isoformat(),
            'duration': str(training_duration),
            'total_parameters': total_params,
            'trainable_parameters': trainable_params,
            'final_model_path': final_model_path,
            'lora_adapter_path': lora_adapter_path,
            'device': str(device),
            'training_args': {
                'num_epochs': training_config['training_args'].num_train_epochs,
                'batch_size': training_config['training_args'].per_device_train_batch_size,
                'learning_rate': training_config['training_args'].learning_rate,
                'weight_decay': training_config['training_args'].weight_decay,
            }
        }
        
        summary_file = os.path.join(training_config['log_dir'], 'training_summary.json')
        with open(summary_file, 'w') as f:
            json.dump(training_summary, f, indent=2)
        
        logger.info("Training completed successfully!")
        logger.info(f"Training summary saved to: {summary_file}")
        
        return trainer, training_config['output_dir']
        
    except Exception as e:
        logger.error(f"Error during training: {e}")
        logger.error(f"Error type: {type(e).__name__}")
        import traceback
        logger.error(f"Traceback: {traceback.format_exc()}")
        return None, None



## Evaluate Intern VL3 Model

In [17]:
# Run InternVL3 Training
print("🚀 Starting InternVL3 LoRA Training...")
print("This may take several hours depending on your hardware.")
print("Check the logs directory for detailed progress information.")

try:
    # Start training
    trainer, output_dir = train_internvl3_with_lora()
    
    if trainer is not None:
        print("✅ Training completed successfully!")
        print(f"📁 Model saved to: {output_dir}")
        print("📊 Check the logs directory for detailed training metrics")
        
        # Clear memory after training
        clear_memory()
        
    else:
        print("❌ Training failed. Check the logs for details.")
        
except Exception as e:
    print(f"❌ Training failed with error: {e}")
    print("📋 Check the detailed logs in the logs directory for troubleshooting")
    import traceback
    traceback.print_exc()

🚀 Starting InternVL3 LoRA Training...
This may take several hours depending on your hardware.
Check the logs directory for detailed progress information.
Training set: 800 examples
Validation set: 200 examples
❌ Training failed with error: CUDA out of memory. Tried to allocate 462.00 MiB. GPU 0 has a total capacity of 15.70 GiB of which 194.94 MiB is free. Including non-PyTorch memory, this process has 15.49 GiB memory in use. Of the allocated memory 15.03 GiB is allocated by PyTorch, and 234.97 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)
📋 Check the detailed logs in the logs directory for troubleshooting
❌ Training failed with error: CUDA out of memory. Tried to allocate 462.00 MiB. GPU 0 has a total capacity of 15.70 GiB of which 194.94 MiB is f

Traceback (most recent call last):
  File "/tmp/ipykernel_16767/548338131.py", line 8, in <module>
    trainer, output_dir = train_internvl3_with_lora()
  File "/tmp/ipykernel_16767/2753707067.py", line 623, in train_internvl3_with_lora
    model = model.to(device)
  File "/home/malneyugnfl/anaconda3/envs/huggingface/lib/python3.10/site-packages/torch/nn/modules/module.py", line 1369, in to
    return self._apply(convert)
  File "/home/malneyugnfl/anaconda3/envs/huggingface/lib/python3.10/site-packages/torch/nn/modules/module.py", line 928, in _apply
    module._apply(fn)
  File "/home/malneyugnfl/anaconda3/envs/huggingface/lib/python3.10/site-packages/torch/nn/modules/module.py", line 928, in _apply
    module._apply(fn)
  File "/home/malneyugnfl/anaconda3/envs/huggingface/lib/python3.10/site-packages/torch/nn/modules/module.py", line 928, in _apply
    module._apply(fn)
  [Previous line repeated 6 more times]
  File "/home/malneyugnfl/anaconda3/envs/huggingface/lib/python3.10/site-pa

In [ ]:
# Run InternVL3 Evaluation
print("📊 Starting InternVL3 Evaluation on Test Data...")
print("This will evaluate the trained model on the full test set.")

try:
    # Run evaluation
    evaluation_results = evaluate_internvl3_on_test()
    
    if evaluation_results:
        print("✅ Evaluation completed successfully!")
        print("📈 Results Summary:")
        print(f"   Overall Accuracy: {evaluation_results['accuracy']:.4f}")
        print(f"   Total Examples: {evaluation_results['total_processed']}")
        print(f"   Correct Predictions: {evaluation_results['correct_predictions']}")
        
        print(f"\n📋 Per-Question-Type Accuracy:")
        for q_type, accuracy in evaluation_results['question_type_accuracy'].items():
            print(f"   {q_type}: {accuracy:.4f}")
        
        print("📁 Detailed results saved to logs directory")
        
        # Clear memory after evaluation
        clear_memory()
        
    else:
        print("❌ Evaluation failed. Check the logs for details.")
        
except Exception as e:
    print(f"❌ Evaluation failed with error: {e}")
    print("📋 Check the detailed logs in the logs directory for troubleshooting")
    import traceback
    traceback.print_exc()

In [ ]:
# Evaluation function
def evaluate_internvl3_on_test(log_dir=None):
    """Evaluate trained InternVL3 model on test data"""
    
    # Setup evaluation logging if not already setup
    if not log_dir:
        _, log_dir = setup_logging("./logs", "internvl3_evaluation")
    
    # Get logger
    eval_logger = logging.getLogger('internvl3_evaluation')
    
    eval_logger.info("="*60)
    eval_logger.info("STARTING INTERNVL3 EVALUATION ON TEST DATA")
    eval_logger.info("="*60)
    
    # Get test data
    test_data = get_evaluation_data()
    eval_logger.info(f"Test data loaded: {len(test_data)} examples")
    
    # Log test data statistics
    question_types = {}
    answer_types = {}
    for example in test_data:
        q_type = example['question_type']
        a_type = example['answer_type']
        question_types[q_type] = question_types.get(q_type, 0) + 1
        answer_types[a_type] = answer_types.get(a_type, 0) + 1
    
    eval_logger.info(f"Test data question types: {question_types}")
    eval_logger.info(f"Test data answer types: {answer_types}")
    
    # Evaluate model
    global model
    eval_start_time = datetime.now()
    eval_logger.info(f"Starting evaluation at: {eval_start_time}")
    
    evaluation_results = evaluate_model_on_mathvista(
        model=model,
        tokenizer=None,
        data=test_data,
        model_name="InternVL3-LoRA",
        log_dir=log_dir
    )
    
    # Log evaluation completion
    eval_end_time = datetime.now()
    eval_duration = eval_end_time - eval_start_time
    eval_logger.info(f"Evaluation completed at: {eval_end_time}")
    eval_logger.info(f"Total evaluation time: {eval_duration}")
    
    # Log final results
    eval_logger.info("="*60)
    eval_logger.info("EVALUATION RESULTS SUMMARY")
    eval_logger.info("="*60)
    eval_logger.info(f"Overall Accuracy: {evaluation_results['accuracy']:.4f}")
    eval_logger.info(f"Total Examples: {evaluation_results['total_processed']}")
    eval_logger.info(f"Correct Predictions: {evaluation_results['correct_predictions']}")
    
    # Log per-question-type accuracy
    eval_logger.info("Per-Question-Type Accuracy:")
    for q_type, accuracy in evaluation_results['question_type_accuracy'].items():
        eval_logger.info(f"  {q_type}: {accuracy:.4f}")
    
    # Save results with enhanced information
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    results_file = os.path.join(log_dir, f"internvl3_evaluation_{timestamp}.json")
    
    # Add evaluation metadata
    evaluation_results['evaluation_metadata'] = {
        'start_time': eval_start_time.isoformat(),
        'end_time': eval_end_time.isoformat(),
        'duration': str(eval_duration),
        'model_name': 'InternVL3-LoRA',
        'test_dataset_size': len(test_data),
        'device': str(device)
    }
    
    with open(results_file, 'w') as f:
        json.dump(evaluation_results, f, indent=2, default=str)
    
    eval_logger.info(f"Detailed evaluation results saved to: {results_file}")
    
    # Save a simple summary file
    summary_file = os.path.join(log_dir, f"evaluation_summary_{timestamp}.txt")
    with open(summary_file, 'w') as f:
        f.write("InternVL3-LoRA Evaluation Summary\n")
        f.write("="*50 + "\n")
        f.write(f"Evaluation Date: {eval_end_time.strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Total Examples: {evaluation_results['total_processed']}\n")
        f.write(f"Overall Accuracy: {evaluation_results['accuracy']:.4f}\n")
        f.write(f"Evaluation Duration: {eval_duration}\n")
        f.write("\nPer-Question-Type Results:\n")
        for q_type, accuracy in evaluation_results['question_type_accuracy'].items():
            f.write(f"  {q_type}: {accuracy:.4f}\n")
    
    eval_logger.info(f"Evaluation summary saved to: {summary_file}")
    
    return evaluation_results

print("InternVL3 training and evaluation setup completed with enhanced logging!")
print("="*70)
print("AVAILABLE FUNCTIONS:")
print("="*70)
print("🚀 train_internvl3_with_lora(): Train the model with LoRA")
print("📊 evaluate_internvl3_on_test(): Evaluate on test data")
print("⚙️  setup_internvl3_training(): Setup training configuration")
print("📝 setup_logging(): Initialize logging system")
print("💾 log_system_info(): Log system information")
print("📈 log_training_config(): Log training configuration")
print("")
print("LOGGING FEATURES:")
print("✓ Comprehensive training progress logging")
print("✓ System and model information logging")
print("✓ Performance metrics tracking")
print("✓ Error logging with tracebacks")
print("✓ Memory usage monitoring")
print("✓ Separate logs for different components")
print("✓ Training and evaluation summaries")
print("✓ JSON metrics export for analysis")
print("")
print("All logs will be saved to the './logs' directory with timestamped folders.")


## Import SophiaVL-R1 Model

In [ ]:
## Source: https://huggingface.co/bunny127/SophiaVL-R1-Thinking-Reward-Model-3B

# Load model directly
from transformers import AutoProcessor, AutoModelForVision2Seq
from peft import LoraConfig, get_peft_model, TaskType
import requests
from PIL import Image
import torch

# Load processor and model
processor = AutoProcessor.from_pretrained("bunny127/SophiaVL-R1-Thinking-Reward-Model-3B")
model = AutoModelForVision2Seq.from_pretrained("bunny127/SophiaVL-R1-Thinking-Reward-Model-3B")

# Define LoRA configuration
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,  # Vision-to-Text is similar to sequence-to-sequence
    inference_mode=False,  # Set to True for inference only
    r=16,  # Rank of the adaptation
    lora_alpha=32,  # LoRA scaling parameter
    lora_dropout=0.1,  # Dropout for LoRA layers
    target_modules=[  # Target specific modules in the model
        "q_proj",
        "v_proj", 
        "k_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    bias="none",  # Don't train bias parameters
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

# Move model to device
model = model.to(device)



# Download and resize image to reduce token count
image_url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"
image = Image.open(requests.get(image_url, stream=True).raw)

# Resize image to reduce token count (smaller image = fewer tokens)
max_size = 512  # Reduce from default to limit tokens
image = image.resize((max_size, int(max_size * image.height / image.width)), Image.Resampling.LANCZOS)

print(f"Resized image to: {image.size}")

messages = [
    {
        "role": "user", 
        "content": [
            {"type": "image", "image": image},  # Use PIL image directly
            {"type": "text", "text": "What animal is on the candy?"}
        ]
    },
]

# Process with truncation to handle long sequences
try:
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        max_length=9500,  # Leave some room for generation
        truncation=True,  # Truncate if too long
    ).to(model.device)
    
    print(f"Input sequence length: {inputs['input_ids'].shape[-1]} tokens")
    
    # Generate with shorter output to stay within limits
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=60,  # Reduced from 40
            do_sample=False,
            pad_token_id=processor.tokenizer.eos_token_id
        )
    
    # Decode only the generated part
    generated_text = processor.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    print(f"Generated response: {generated_text}")
    
except Exception as e:
    print(f"Error during inference: {e}")
    print("The sequence is still too long even after resizing. Try with an even smaller image or shorter text.")

## Fine Tune SophiaVL-R1 Model

In [ ]:
# Fine Tune Intern VL3 Model w/LorA on TestMini Subset